In [1]:
# app/app.py
import gradio as gr
import joblib
import pandas as pd
import numpy as np
import json
import os

print("="*60)
print("LOADING MODEL AND PREPROCESSORS")
print("="*60)

# Load model and preprocessors
model = joblib.load('../models/xgb_final_model.joblib')
feature_names = joblib.load('../models/feature_names.joblib')
scaler = joblib.load('../models/scaler.joblib')

# Load metadata
with open('../models/metadata.json', 'r') as f:
    metadata = json.load(f)

print(f"✅ Model loaded: {metadata['model_name']}")
print(f"✅ Features: {len(feature_names)}")
print(f"✅ AUC-ROC: {metadata['metrics']['auc_roc']:.4f}")

LOADING MODEL AND PREPROCESSORS
✅ Model loaded: XGBoost
✅ Features: 27
✅ AUC-ROC: 0.8495


In [2]:
def predict_loan_default(loanamount_x, totaldue_x, termdays_x, longitude_gps, latitude_gps,
                         loanamount_y, totaldue_y, termdays_y, employment_status,
                         bank_name, bank_account_type):
    """
    Predict loan default probability based on customer inputs.
    """
    # Map employment status to ordinal value
    emp_mapping = {
        'Permanent': 6,
        'Contract': 5,
        'Self-Employed': 4,
        'Student': 3,
        'Retired': 2,
        'Unemployed': 1
    }
    emp_encoded = emp_mapping.get(employment_status, 1)
    
    # Create feature dictionary with all zeros
    input_dict = {col: 0 for col in feature_names}
    
    # Fill in numeric features
    input_dict['loanamount_x'] = loanamount_x
    input_dict['totaldue_x'] = totaldue_x
    input_dict['termdays_x'] = termdays_x
    input_dict['longitude_gps'] = longitude_gps
    input_dict['latitude_gps'] = latitude_gps
    input_dict['loanamount_y'] = loanamount_y
    input_dict['totaldue_y'] = totaldue_y
    input_dict['termdays_y'] = termdays_y
    input_dict['employment_status_clients'] = emp_encoded
    
    # One-hot encoding for bank_name
    if bank_name != 'None':
        bank_col = f'bank_name_clients_{bank_name}'
        if bank_col in input_dict:
            input_dict[bank_col] = 1
    
    # One-hot encoding for bank_account_type
    if bank_account_type != 'None':
        account_col = f'bank_account_type_{bank_account_type}'
        if account_col in input_dict:
            input_dict[account_col] = 1
    
    # Convert to DataFrame
    input_df = pd.DataFrame([input_dict])
    
    # Ensure correct column order
    input_df = input_df[feature_names]
    
    # Predict
    prediction = model.predict(input_df)[0]
    probability = model.predict_proba(input_df)[0]
    
    # Results
    if prediction == 1:
        status = "✅ Good (Will Repay)"
        confidence = probability[1] * 100
    else:
        status = "❌ Bad (Default Risk)"
        confidence = probability[0] * 100
    
    return f"""
    📊 Prediction: {status}
    📈 Confidence: {confidence:.1f}%
    
    📝 Risk Score: {probability[0]:.3f} (0=Low Risk, 1=High Risk)
    """

In [3]:
# Create interface
iface = gr.Interface(
    fn=predict_loan_default,
    inputs=[
        gr.Number(label="Current Loan Amount (NGN)", value=30000),
        gr.Number(label="Total Due (NGN)", value=33000),
        gr.Dropdown(choices=[15, 30, 60, 90], label="Current Loan Term (days)", value=30),
        gr.Number(label="Longitude GPS", value=3.5),
        gr.Number(label="Latitude GPS", value=6.5),
        gr.Number(label="Previous Loan Amount (NGN)", value=10000),
        gr.Number(label="Previous Total Due (NGN)", value=13000),
        gr.Dropdown(choices=[15, 30, 60, 90], label="Previous Loan Term (days)", value=30),
        gr.Dropdown(
            choices=['Permanent', 'Self-Employed', 'Student', 'Unemployed', 'Retired', 'Contract'],
            label="Employment Status",
            value='Permanent'
        ),
        gr.Dropdown(
            choices=['GT Bank', 'First Bank', 'Access Bank', 'UBA', 'Diamond Bank', 
                     'Zenith Bank', 'Stanbic IBTC', 'EcoBank', 'FCMB', 'Skye Bank',
                     'Fidelity Bank', 'Sterling Bank', 'Wema Bank', 'Heritage Bank',
                     'Keystone Bank', 'Union Bank', 'Standard Chartered', 'Unity Bank'],
            label="Bank Name",
            value='GT Bank'
        ),
        gr.Dropdown(
            choices=['Savings', 'Current', 'Other'],
            label="Bank Account Type",
            value='Savings'
        )
    ],
    outputs=gr.Textbox(label="Prediction Result", lines=6),
    title="🏦 SuperLender Loan Default Predictor",
    description="""
    Enter customer details below to predict if they will repay or default on their loan.
    
    **Note:** This model uses XGBoost with 0.8495 AUC-ROC.
    """,
    examples=[
        [30000, 33000, 30, 3.5, 6.5, 10000, 13000, 30, 'Permanent', 'GT Bank', 'Savings'],
        [50000, 55000, 60, 3.5, 6.5, 20000, 26000, 30, 'Unemployed', 'Access Bank', 'Current'],
        [10000, 11500, 15, 3.5, 6.5, 5000, 6500, 15, 'Self-Employed', 'First Bank', 'Other']
    ]
)

# Launch
if __name__ == "__main__":
    iface.launch(share=True)

* Running on local URL:  http://127.0.0.1:7860

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.


C:\Users\blaizo\anaconda3\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
C:\Users\blaizo\anaconda3\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
